# Iterative imputer (MICE)

Note ML-039. We fill three gaps in a 5-row table by hand, one column at a time, then let scikit-learn's `IterativeImputer` do the same, and finally use it on the full 50-row data with a train/test split.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
# IterativeImputer is still experimental: this import switches it on
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split

## 1. The toy table

50 startups: money spent on R&D, administration and marketing (in dollars). We divide by 10,000, round, and keep 5 random rows and the three spend columns (the target `Profit` is dropped: we only impute input columns).

In [ ]:
full = pd.read_csv("data/50_Startups.csv")
cols = ["R&D Spend", "Administration", "Marketing Spend"]
# units of 10,000 dollars, rounded to whole numbers
df = np.round(full[cols] / 10000)
# 5 random rows (seed 9)
df = df.sample(5, random_state=np.random.RandomState(9)).reset_index(drop=True)
df

In [ ]:
# hide one value in each column and remember where the gaps are
truth = df.copy()
df.iloc[1, 0] = np.nan
df.iloc[3, 1] = np.nan
df.iloc[4, 2] = np.nan
gaps = df.isna()
df

## 2. Step 0: fill every gap with its column mean

In [ ]:
df0 = df.fillna(df.mean())
df0

## 3. One iteration by hand

For each column, left to right: put its gaps back to NaN, train a linear regression on the rows without a gap (this column from the other two), predict the gaps. Values are cut to 2 decimals, as in a hand calculation.

In [ ]:
def one_iteration(prev, decimals=2):
    # start from the previous table; the fills change as we go
    cur = prev.copy()
    for col in cols:
        rows = gaps[col]
        others = [c for c in cols if c != col]
        # train on the rows where this column was never missing
        model = LinearRegression().fit(cur.loc[~rows, others], cur.loc[~rows, col])
        pred = model.predict(cur.loc[rows, others])
        # cut (not round) to 2 decimals, like the hand calculation
        cur.loc[rows, col] = np.trunc(pred * 10**decimals) / 10**decimals if decimals else pred
    return cur

df1 = one_iteration(df0)
df1

In [ ]:
# change since iteration 0: zero everywhere except the three gaps
df1 - df0

In [ ]:
df2 = one_iteration(df1)
df2 - df1

## 4. Iterate until nothing changes

Cutting to 2 decimals makes the numbers drift a little, so from here we keep full precision.

In [ ]:
cur = df0.copy()
for it in range(1, 21):
    new = one_iteration(cur, decimals=None)
    change = (new - cur).abs().max().max()
    print(it, new.values[gaps.values].round(2), "largest change", round(change, 4))
    cur = new
    if change < 0.001:
        break

In [ ]:
# the true hidden values: 4, 16, 3
truth.values[gaps.values]

## 5. The same with scikit-learn

`IterativeImputer` with `LinearRegression` gives the same numbers at full precision. `tol=0` forces exactly `max_iter` iterations.

In [ ]:
warnings.filterwarnings("ignore", message=".*Early stopping.*")
for m in [1, 2, 10]:
    imp = IterativeImputer(estimator=LinearRegression(), max_iter=m, tol=0)
    print(m, imp.fit_transform(df)[gaps.values].round(2))

In [ ]:
# defaults: BayesianRidge, max_iter=10, tol=0.001, initial_strategy="mean"
imp = IterativeImputer(random_state=0)
print(imp.fit_transform(df)[gaps.values].round(2), "stopped after", imp.n_iter_, "iterations")

## 6. Real use: fit on the training set only

All 50 rows. We split 70/30, hide 20% of the values in both parts at random, fit each imputer on the training part and fill the test part. We compare the filled values with the true ones (root mean squared error, in units of 10,000 dollars), averaged over 100 random splits.

In [ ]:
data = full[cols] / 10000
imputers = {"mean": SimpleImputer(), "KNN (k=5)": KNNImputer(), "iterative": IterativeImputer(random_state=0)}
sq_err = {name: [] for name in imputers}
for seed in range(100):
    train, test = train_test_split(data, test_size=0.3, random_state=seed)
    rng = np.random.default_rng(seed)
    # hide 20% of the values in each part
    train_gaps = train.mask(rng.random(train.shape) < 0.2)
    test_gaps = test.mask(rng.random(test.shape) < 0.2)
    hidden = test_gaps.isna().values
    for name, imp in imputers.items():
        # learn from the training set only, then fill the test set
        imp.fit(train_gaps)
        filled = imp.transform(test_gaps)
        sq_err[name] += list((filled - test.values)[hidden] ** 2)
{name: round(np.sqrt(np.mean(e)), 2) for name, e in sq_err.items()}

## 7. Several imputations

`sample_posterior=True` draws each fill from the model's spread instead of taking its best guess, so different seeds give different filled tables. This is the 'multiple' in MICE.

In [ ]:
for seed in range(3):
    imp = IterativeImputer(sample_posterior=True, random_state=seed)
    print(seed, imp.fit_transform(df)[gaps.values].round(2))

In [ ]:
# Test: does the iterative imputer win because the columns are related?
# Shuffle each column on its own (keeps its values, breaks the links) and rerun the same 100 splits
def compare(data):
    imputers = {"mean": SimpleImputer(), "KNN (k=5)": KNNImputer(), "iterative": IterativeImputer(random_state=0)}
    sq_err = {name: [] for name in imputers}
    for seed in range(100):
        train, test = train_test_split(data, test_size=0.3, random_state=seed)
        rng = np.random.default_rng(seed)
        train_gaps = train.mask(rng.random(train.shape) < 0.2)
        test_gaps = test.mask(rng.random(test.shape) < 0.2)
        hidden = test_gaps.isna().values
        for name, imp in imputers.items():
            imp.fit(train_gaps)
            sq_err[name] += list((imp.transform(test_gaps) - test.values)[hidden] ** 2)
    return {name: round(np.sqrt(np.mean(e)), 2) for name, e in sq_err.items()}
rng = np.random.default_rng(0)
shuffled = data.apply(lambda c: rng.permutation(c.values))
print(shuffled.corr().round(2))
compare(shuffled)